In [ ]:
!pip install transformers==4.28.0
!pip install datasets

In [ ]:
import re
import string
import tensorflow as tf

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer
from transformers import AdamW, get_linear_schedule_with_warmup, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [ ]:
train = pd.read_csv("/content/train_dataset.csv")
validation = pd.read_csv("/content/validation_dataset.csv")
test = pd.read_csv("/content/test_dataset.csv")

In [ ]:
import datasets
import pandas as pd
from datasets import Dataset, DatasetDict

train_df = pd.DataFrame({
     "text" : train["Translated_Text"],
     "labels" : train['label']

})
val_df = pd.DataFrame({
     "text" : validation["Translated_Text"],
     "labels" : validation['label']

})

test_df = pd.DataFrame({
     "text" : test["Translated_Text"],
     "labels" : test['label']

})

train_dataset = Dataset.from_dict(train_df)
val_dataset = Dataset.from_dict(val_df)
test_dataset = Dataset.from_dict(test_df)
dataset = datasets.DatasetDict({"train":train_dataset,"validation":val_dataset,"test":test_dataset})

In [ ]:
dataset

In [ ]:
dataset.num_rows

In [ ]:
dataset['train'][0]

In [ ]:
train.head()

In [ ]:
test.head()

In [ ]:
validation.head()

#Hyperparameters

In [ ]:
num_epoch = 10
sq_len = 128
batch_size = 32
lr_rate = 2e-5
epsilon = 1e-8
hidden_dropout = 0.05
warmup_ratio = 0.06
weight_decay = 0.01

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('setu4993/LaBSE')

def encode_batch(batch):
  """Encodes a batch of input data using the model tokenizer."""
  return tokenizer(batch["text"], max_length=sq_len, truncation=True, padding="max_length")

# Encode the input data
dataset = dataset.map(encode_batch, batched=True)
# The transformers model expects the target class column to be named "labels"
#dataset.rename_column_("label", "labels")
# Transform to pytorch tensors and only output the required columns
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

In [ ]:
# Calculation of the maximum length of any text in terms of tokens

max_len = 0

# Concatenate all datasets
all_texts_df = pd.concat([train["Translated_Text"], validation["Translated_Text"], test["Translated_Text"]])

# Iterate through each text in the dataset
for text in all_texts_df:
    input_ids = tokenizer.encode(text, add_special_tokens=True)
    max_len = max(max_len, len(input_ids))

print(f"The maximum length of any text in terms of tokens is: {max_len}")


In [ ]:
dataset

## Training

In [ ]:
from transformers import AutoConfig, AutoModelForSequenceClassification

# Model and training configuration
config = AutoConfig.from_pretrained('setu4993/LaBSE',
                                    num_labels=2,
                                    problem_type="single_label_classification")

config.hidden_dropout_prob = hidden_dropout
model = AutoModelForSequenceClassification.from_pretrained('setu4993/LaBSE', config=config)

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
# Define the function to compute the metrics during evaluation
from sklearn.metrics import accuracy_score, precision_recall_fscore_support,precision_score,recall_score,confusion_matrix
import time
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average='weighted')
    precision = precision_score(labels, preds, average='weighted',zero_division=0)
    recall = recall_score(labels, preds, average='weighted',zero_division=0)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    return {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall, 'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn}



In [ ]:
optimizer = AdamW(model.parameters(), lr=lr_rate, eps=epsilon)
total_steps = len(dataset["train"]) * num_epoch
warmup_steps = int(total_steps * warmup_ratio)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

In [ ]:
from datetime import datetime
import pandas as pd

# Create empty lists to store results
epoch_list = []
training_loss_list = []
validation_loss_list = []
training_time_list = []
validation_time_list = []
validation_accuracy_list = []
validation_f1_list = []

test_tp_list = []
test_tn_list = []
test_fp_list = []
test_fn_list = []

test_accuracy_list = []
test_f1_list = []
test_precision_list = []
test_recall_list = []
# Define the trainer Arguments
training_args = TrainingArguments(
          output_dir='./results',
          overwrite_output_dir='True',
          num_train_epochs=1,
          per_device_train_batch_size=batch_size,
          per_device_eval_batch_size=batch_size,
          warmup_ratio=warmup_ratio,
          weight_decay=weight_decay,
          logging_dir='./logs',
          logging_steps=100,
          save_strategy='no',
          evaluation_strategy='no',
          load_best_model_at_end=True,
          metric_for_best_model='accuracy',
          greater_is_better=True,


    )

train_epoch = num_epoch
# Train the model and track the metrics after each epoch
for epoch in range(train_epoch):


    # Define the trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset["train"],
        eval_dataset=dataset["validation"],
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        # optimizers=(optimizer, None)
    )


    # Train the model for one epoch
    start_time = datetime.now()
    train_result = trainer.train()
    end_time = datetime.now()

    # Perform optimization steps
    optimizer.step()


    # Clear gradients
    optimizer.zero_grad()

    # Perform evaluation on the validation set
    validation_result = trainer.evaluate(eval_dataset=dataset["validation"])

    # Perform evaluation on the test set
    test_result = trainer.evaluate(eval_dataset=dataset["test"])


    # Extract the desired information
    training_loss = train_result.training_loss
    validation_loss = validation_result["eval_loss"]

    validation_time = validation_result["eval_runtime"]
    training_time = train_result.metrics["train_runtime"]

    validation_accuracy = validation_result["eval_accuracy"]
    validation_f1 = validation_result["eval_f1"]


    test_accuracy = test_result["eval_accuracy"]
    test_f1 = test_result["eval_f1"]
    test_precision = test_result["eval_precision"]
    test_recall = test_result["eval_recall"]

    test_tp = test_result["eval_tp"]
    test_tn = test_result["eval_tn"]
    test_fp = test_result["eval_fp"]
    test_fn = test_result["eval_fn"]

    val_runtime_formatted = pd.to_datetime(validation_time, unit='s').strftime('%H:%M:%S')
    train_runtime_formatted = pd.to_datetime(training_time, unit='s').strftime('%H:%M:%S')

    # Append the metrics to the respective lists
    epoch_list.append(epoch + 1)
    training_loss_list.append(training_loss)
    validation_loss_list.append(validation_loss)

    validation_accuracy_list.append(validation_accuracy)
    validation_f1_list.append(validation_f1)

    training_time_list.append(train_runtime_formatted)
    validation_time_list.append(val_runtime_formatted)

    test_tp_list.append(test_tp)
    test_tn_list.append(test_tn)
    test_fp_list.append(test_fp)
    test_fn_list.append(test_fn)


    test_accuracy_list.append(test_accuracy)
    test_f1_list.append(test_f1)
    test_precision_list.append(test_precision)
    test_recall_list.append(test_recall)

# Create a DataFrame to store the results
train_df = pd.DataFrame({
    "Epoch": epoch_list,
    "Training Loss": training_loss_list,
    "Validation Loss": validation_loss_list,
    "Validation Accuracy": validation_accuracy_list,
    "Validation F1": validation_f1_list,
    "Training Time": training_time_list,
    "Validation Time": validation_time_list,
})

test_df = pd.DataFrame({
    "Epoch": epoch_list,
    "Test TP": test_tp_list,
    "Test TN": test_tn_list,
    "Test FP": test_fp_list,
    "Test FN": test_fn_list,
    "Test Accuracy": test_accuracy_list,
    "Test F1": test_f1_list,
    "Test Precision": test_precision_list,
    "Test Recall": test_recall_list,
})

In [ ]:
train_df.head(10)

In [ ]:
test_df.head(10)

#Store Results in Excel File

In [ ]:
train_df.to_excel("Train LaBSE-128SL-32BS-0.05HD-0.06WR-0.01WD.xlsx")

In [ ]:
test_df.to_excel("Test LaBSE-128SL-32BS-0.05HD-0.06WR-0.01WD.xlsx")

# Attention Visualization

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import matplotlib.pyplot as plt
import seaborn as sns

# Load again the model and tokenizer with attention output enabled
tokenizer = AutoTokenizer.from_pretrained('setu4993/LaBSE')
model = AutoModelForSequenceClassification.from_pretrained('setu4993/LaBSE', output_attentions=True)

# Evaluation mode
model.eval()

# Sample sentence
sentence = "αυτοί που πληρώνουν τους φόρους μας συνεχίζουν να μας δείχνουν τα μάτια βγάλτε τους τα μάτια πάρτε τα και πετάξτε τα έξω από τα ίδια ιδρύματα" # Hate speech sentence from train dataset
inputs = tokenizer(sentence, return_tensors='pt', padding=True, truncation=True, max_length=128)


In [ ]:

# Transfer inputs to the same device of the model
device = next(model.parameters()).device
inputs = {k: v.to(device) for k, v in inputs.items()}

# Forward pass
with torch.no_grad():
    outputs = model(**inputs)
    attentions = outputs.attentions

# Flatten the attentions
attention = torch.cat([att for layer in attentions for att in layer], dim=0)
attention = attention.mean(dim=0)

In [ ]:
# Convert attention matrix to numpy
attention_matrix = attention.cpu().numpy()

# Convert input IDs to tokens for labeling axes
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'].squeeze().tolist())

In [ ]:
# Visualization of attention weights
plt.figure(figsize=(10, 8))
sns.heatmap(attention_matrix, xticklabels=tokens, yticklabels=tokens, cmap='viridis')
plt.title('Average Attention Weights Across All Heads and Layers')
plt.xlabel('Tokens in Sequence')
plt.ylabel('Tokens in Sequence')
plt.show()

For example, ##τα most likely stands for either a suffix or a part of some bigger word which has been split by the tokenizer for better comprehension. The bright spots such as that linking ##φορούς with ##μας represent regions where attention concentrates. This is what the sentence means in terms of hate speech as perceived by the model.

#Counterfactual Explanations

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import pandas as pd
import shap
import scipy as sp

# Load again tokenizer and model
tokenizer = AutoTokenizer.from_pretrained('setu4993/LaBSE')
model = AutoModelForSequenceClassification.from_pretrained('setu4993/LaBSE')
model.eval()

# Check the availability of CUDA
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
# Prediction function
def f(x):
    tv = torch.tensor(
        [
            tokenizer.encode(v, padding="max_length", max_length=128, truncation=True)
            for v in x
        ]
    ).cuda()
    outputs = model(tv)[0].detach().cpu().numpy()
    scores = (np.exp(outputs).T / np.exp(outputs).sum(-1)).T
    return sp.special.logit(scores[:, 1])

# Πrediction function for pre-encoded inputs
def predict(inputs):
    inputs = {key: value.to(device) for key, value in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        probabilities = F.softmax(outputs.logits, dim=-1)
        predictions = torch.argmax(probabilities, dim=-1)
    return probabilities, predictions

# Example sentences for prediction
text = "σταματήστε τις ανοησίες σήμερα ο λαός της ινδίας υποφέρει από αυτό το σκυλί οι μουσουλμάνοι του πακιστάν και ο λαός της ινδίας αντιμετωπίζουν προβλήματα εξαιτίας αυτού του ανθρώπου"  # original hate speech sentence from train dataset
counterfactual_text = "ας εστιάσουμε στα σημαντικά θέματα σήμερα οι κοινότητες τόσο στην ινδία όσο και στο πακιστάν αντιμετωπίζουν σοβαρές προκλήσεις οι οποίες χρήζουν της κοινής μας προσοχής και κατανόησης"  # it's similar to the original one but without hate speech


In [ ]:
# SHAP explainer initialization
explainer = shap.Explainer(f, tokenizer)

# Calculate SHAP values
shap_values = explainer([text, counterfactual_text])

# Visualize the explanation for both texts
shap.plots.text(shap_values[0])
shap.plots.text(shap_values[1])

In [ ]:
# Bar plot of Shap values
shap.plots.bar(shap_values[0]) # for instance 0

In [ ]:
# Bar plot of Shap values
shap.plots.bar(shap_values[1]) # for instance 1

In [ ]:
# Encode both the original and counterfactual texts
encoded_input = tokenizer(text, return_tensors='pt', max_length=512, truncation=True, padding='max_length').to(device)
encoded_counterfactual = tokenizer(counterfactual_text, return_tensors='pt', max_length=512, truncation=True, padding='max_length').to(device)

# Predictions and probabilities for the original and counterfactual texts
original_probabilities, original_predictions = predict(encoded_input)
counterfactual_probabilities, counterfactual_predictions = predict(encoded_counterfactual)


In [ ]:
# Prepare data for saving
data = {
    "Text": ["Original text", "Counterfactual text"],
    "Probabilities": [original_probabilities.tolist(), counterfactual_probabilities.tolist()],
    "Predicted Class": [original_predictions.item(), counterfactual_predictions.item()]
}

# Creation of a DataFrame
results_df = pd.DataFrame(data)

# Save to a CSV file
results_df.to_csv('model_predictions.csv', index=False)
